In [1]:
!pip cache purge --quiet

In [4]:
# !pip install langchain --quiet
# !pip install langchain-community --quiet
# !pip install langchain-ollama --quiet
# !pip install ollama --quiet
# !pip install pdf2image --quiet
# !pip install pdfminer.six --quiet
# !pip install unstructured==0.10.14 --quiet

In [8]:
import re
import nltk
import ollama
from langchain.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain.document_loaders import OnlinePDFLoader
from langchain.vectorstores.utils import DistanceStrategy
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
embeddings_model = "all-minilm"
ollama.pull(embeddings_model)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [10]:
llm = 'deepseek-r1:1.5b'
ollama.pull(llm)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [11]:
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/danishkarur/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/danishkarur/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [12]:
loader = OnlinePDFLoader("https://www.investni.com/sites/default/files/2021-02/NI-fintech-document.pdf")
data = loader.load()

In [13]:
print(f"You have {len(data)} document(s) in your data")
print(f"There are {len(data[0].page_content)} characters in your document")

You have 1 document(s) in your data
There are 39674 characters in your document


In [15]:
# no.of characters diveded by 2000 (chunk_size) = Total no of pages
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 20
)
texts = text_splitter.split_documents(data)
print(f"You have {len(texts)} pages")

You have 23 pages


In [19]:
embeddings = OllamaEmbeddings(
    model = embeddings_model,
)
dimensions = len(embeddings.embed_query(texts[0].page_content))
print("Dimension of Embeddings are: ", dimensions)

Dimension of Embeddings are:  384


In [20]:
docsearch = FAISS.from_documents(texts, embeddings)

In [22]:
prompt = "What are the best investment opportunities in Blockchain?"
docs = docsearch.similarity_search(prompt)
data = docs[0].page_content
print(data)

Within our well respected financial and related professional services cluster, global leaders including Deloitte and PwC are currently working on the application of blockchain solutions within insurance, digital banking and cross-border payments.

PwC

Vox Financial Partners

The PwC global blockchain impact centre in Belfast comprises a team of fintech professionals with deep expertise and a proven record of delivery of insurance, banking, e-commerce and bitcoin products and services. The Belfast team is exploring the application of this disruptive technology to digital currencies, digital assets, identity and smart contracts. The specialist team has already delivered a significant proof of concept project for the Bank of England, to investigate the capability of distributed ledger technology.

www.pwc.co.uk

Founded in 2016, the Belfast based Fintech consultancy Vox Financial Partners works with top-tier banks and broker- dealer clients in the US and Europe. Vox offers high quality r

In [24]:
output = ollama.generate(
    model = llm,
    prompt = f"Using this data: \n{data}. \nRespond to this prompt: \n{prompt}.\n\n"
    
)
content = output['response']
remove_think_tags = True
if remove_think_tags:
    content = re.sub("<think>.*?</think>","",content,flags = re.DOTALL)

print(content)



The best investment opportunities in Blockchain technology, based on the provided data and potential strategic expansions, can be summarized as follows:

1. **PwC's Blockchain Impact Centre**:
   - **Opportunities**: Leverage PwC's already impactful work with the Bank of England using distributed ledger technology (DLT). Expand into cross-border payments, smart contracts in insurance, and digital banking advancements.
   - **Investment Strategy**: Target early-stage funding for expansion beyond financial services to explore these areas.

2. **Vox Financial Partners**:
   - **Opportunities**: Develop high-quality regulatory expertise in areas such as digital assets, identity, and secure transactions. Explore integrating Opal tools with other blockchain projects.
   - **Investment Strategy**: Focus on late-stage expansion into regulatory domains like digital asset management and cross-border payments.

3. **Rakuten Blockchain Lab**:
   - **Opportunities**: Expand R&D to focus on areas 